# 🇻🇳 Vietnamese Legal Q&A — SFT Fine-Tuning
## Qwen3.5-35B-A3B (MoE) with Unsloth + QLoRA

**Mục tiêu**: Fine-tune Qwen3.5-35B-A3B trên 7,636 cặp Q&A pháp luật Việt Nam
đã được distill từ Gemini 2.5, kèm reasoning chain `<think>...</think>`.

**Hardware yêu cầu**: A100 40GB / A100 80GB / H100

**Data source**: `data_gen/train.jsonl` (7,636 records), `data_gen/val.jsonl` (954 records)

---

| Thành phần | Chi tiết |
|---|---|
| Base model | `unsloth/Qwen3.5-35B-A3B` (35B total, 3B active — MoE) |
| Technique | QLoRA 4-bit + MoE LoRA |
| Data format | `instruction` + `input` (RAG context) → `output` (với `<think>` reasoning) |
| Chat template | `qwen3-thinking` |
| Loss masking | `train_on_responses_only` — chỉ train phần assistant |

## 1. 📦 Installation

Cài đặt Unsloth, TRL, và các dependencies cần thiết cho MoE model.
Chạy cell này 1 lần duy nhất.

In [1]:
%uv pip uninstall torch torchvision torchaudio
%uv pip install torch==2.9.0 torchvision==0.24.0 torchaudio==2.9.0 \
  --index-url https://download.pytorch.org/whl/cu129


Using Python 3.12.6 environment at: /usr/local
Uninstalled 3 packages in 2.90s
 - torch==2.8.0+cu129
 - torchaudio==2.8.0+cu129
 - torchvision==0.23.0+cu129
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Resolved 30 packages in 664ms
⠙ Preparing packages... (0/6)
⠙ Preparing packages... (0/6)
⠙ Preparing packages... (0/6)
nvidia-nvshmem-cu12 ------------------------------     0 B/118.88 MiB
⠙ Preparing packages... (0/6)
nvidia-nvshmem-cu12 ------------------------------ 16.00 KiB/118.88 MiB
⠙ Preparing packages... (0/6)
nvidia-nvshmem-cu12 ------------------------------ 16.00 KiB/118.88 MiB
nvidia-nccl-cu12 ------------------------------ 16.00 KiB/307.42 MiB
⠙ Preparing packages... (0/6)
nvidia-nvshmem-cu12 ------------------------------ 16.00 KiB/118.88 MiB
nvidia-nccl-cu12 ------------------------------ 16.00 KiB/307.42 MiB
⠙ Preparing packages... (0/6)
nvidia-nvshmem-cu12 ------------------------------ 16.00 KiB/118.8

In [2]:
%uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
%uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
%uv pip install --no-build-isolation flash-linear-attention causal_conv1d
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    %uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
%uv pip install --no-deps --upgrade "torchao"

Using Python 3.12.6 environment at: /usr/local
Resolved 4 packages in 70ms
⠙ Preparing packages... (0/4)
⠙ Preparing packages... (0/4)
⠙ Preparing packages... (0/4)
⠙ Preparing packages... (0/4)
⠙ Preparing packages... (0/4)
⠙ Preparing packages... (0/4)
⠙ Preparing packages... (0/4)
⠙ Preparing packages... (0/4)
⠙ Preparing packages... (0/4)
⠙ Preparing packages... (0/4)
unsloth-zoo ------------------------------     0 B/902.55 KiB
⠙ Preparing packages... (0/4)
unsloth-zoo ------------------------------ 14.83 KiB/902.55 KiB
⠙ Preparing packages... (0/4)
trl        ------------------------------     0 B/532.02 KiB
unsloth-zoo ------------------------------ 14.83 KiB/902.55 KiB
⠙ Preparing packages... (0/4)
trl        ------------------------------ 14.86 KiB/532.02 KiB
unsloth-zoo ------------------------------ 14.83 KiB/902.55 KiB
⠙ Preparing packages... (0/4)
trl        ------------------------------ 14.86 KiB/532.02 KiB
unsloth-zoo ------------------------------ 14.83 KiB/902.55 KiB


In [3]:
import json, platform, sys, torch
from urllib.request import urlopen

py   = f"cp{sys.version_info.major}{sys.version_info.minor}"
tv   = ".".join(torch.__version__.split("+")[0].split(".")[:2])
assert (cu := (torch.version.cuda or "").split(".")[0]), "CUDA-enabled PyTorch required."
abi  = "TRUE" if torch.compiled_with_cxx11_abi() else "FALSE"
plat = "linux_x86_64" if platform.machine().lower() in ("x86_64", "amd64") else "linux_aarch64"
whl  = f"{py}-{py}-{plat}.whl"

def find(repo, tag, match):
    api = f"https://api.github.com/repos/{repo}/releases/tags/{tag}"
    return next((a["browser_download_url"] for a in json.load(urlopen(api))["assets"] if match(a["name"])), None)

cc1d = find("Dao-AILab/causal-conv1d", "v1.6.1.post4",
            lambda n: n.endswith(whl) and f"+cu{cu}torch{tv}" in n and f"cxx11abi{abi}" in n)
assert cc1d, f"No causal-conv1d wheel for torch {torch.__version__}/cu{cu}/{py}/abi{abi}"
fla = find("fla-org/flash-linear-attention", "v0.4.2", lambda n: n.endswith(whl)) \
      or "https://github.com/fla-org/flash-linear-attention/archive/refs/tags/v0.4.2.tar.gz"

%uv pip install --no-deps "$cc1d" "$fla"
%uv pip install datasets peft bitsandbytes
%uv pip uninstall sentence-transformers torchcodec  # torchcodec import broken on Colab

Using Python 3.12.6 environment at: /usr/local
Resolved 2 packages in 2.08s
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
Building flash-linear-attention @ https://github.com/fla-org/flash-linear-attention/
⠹ Preparing packages... (0/2)
Building flash-linear-attention @ https://github.com/fla-org/flash-linear-attention/
⠹ Preparing packages... (0/2)
Building flash-linear-attention @ https://github.com/fla-org/flash-linear-attention/
⠹ Prepari

In [4]:
%uv pip install z3-solver

Using Python 3.12.6 environment at: /usr/local
Resolved 1 package in 470ms
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
⠸ Preparing packages... (0/1)
⠸ Preparing packages... (0/1)
⠸ Preparing packages... (0/1)
⠼ Preparing packages... (0/1)
Prepared 1 package in 614ms
Installed 1 package in 122m

## 3. 🚀 Load Model — Qwen3.5-35B-A3B

- **35B total parameters, 3B active** (MoE architecture — 256 experts)
- QLoRA 4-bit để giảm VRAM
- `max_seq_length = 8192` — đủ cho RAG context + reasoning

In [1]:
import unsloth
import os
os.environ['UNSLOTH_MOE_DISABLE_AUTOTUNE'] = '1'

from unsloth import FastLanguageModel
import torch

max_seq_length = 8192  # Đủ cho RAG context + reasoning chain dài
lora_rank = 32         # 32 cho chất lượng cao trên A100/H100

model, processor = FastLanguageModel.from_pretrained(
    "claspi2509/legal-qwen35-lora",
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    fast_inference = False, 
)
tokenizer = processor.tokenizer

print(f"\n✅ Model loaded successfully")
print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.1: Fast Qwen3_5_MoE patching. Transformers: 5.2.0.
   \\   /|    NVIDIA H200. Num GPUs = 1. Max memory: 139.801 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu129. CUDA: 9.0. CUDA Toolkit: 12.9. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/1026 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/11.4G [00:00<?, ?B/s]


✅ Model loaded successfully
   GPU: NVIDIA H200
   VRAM: 139.8 GB


## 4. 🎯 Attach LoRA Adapters (MoE-aware)

Quan trọng: Phải bao gồm `gate_up_proj` để LoRA tác động lên MoE expert layers.

In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj", "gate_up_proj",  # MoE layers
    ],
    lora_alpha = lora_rank * 2,  # alpha = 2*r — giúp training nhanh hơn
    use_gradient_checkpointing = True,  # Giảm VRAM đáng kể
    random_state = 3407,
    bias = "none",
)

# Đếm trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\n✅ LoRA attached")
print(f"   Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

RuntimeError: Unsloth: You already added LoRA adapters to your model!

## 5. 📚 Load & Process Legal Dataset

### Schema dữ liệu hiện có (`data_gen/train.jsonl`):
```json
{
  "instruction": "Câu hỏi pháp lý",
  "input": "[#1 | Luật X | Điều Y | nguồn: vector+graph]\nNội dung điều khoản...",
  "output": "<think>\n1. Xác định vấn đề...\n</think>\nCâu trả lời chi tiết...",
  "meta": {"law": "...", "article": "...", ...}
}
```

### Chuyển đổi thành conversation format:
```
system: "Bạn là trợ lý pháp lý..."
user: "Tài liệu tham khảo:\n{input}\n\nCâu hỏi: {instruction}"
assistant: "{output}"  ← đã có <think>...</think>
```

In [3]:
import json
import re
import multiprocessing as mp
from datasets import Dataset, concatenate_datasets
from unsloth.chat_templates import get_chat_template

# ============================================================
# Configuration
# ============================================================
RANDOM_SEED = 3407
MAX_CONTEXT_WINDOW = 8192

# Upload data_gen/ folder lên cloud trước khi chạy.
# Nếu chạy trên Kaggle, upload qua Kaggle Datasets rồi copy vào /kaggle/working/
# Nếu chạy trên Colab, upload qua Google Drive hoặc !gdown
TRAIN_PATH = "/mnt/v/data_gen/advanced_train.jsonl"   # 7,636 records
VAL_PATH   = "/mnt/v/data_gen/advanced_val.jsonl"     # 954 records

SYSTEM_PROMPT = (
    "Bạn là một trợ lý pháp lý chuyên về luật Việt Nam. "
    "Hãy phân tích câu hỏi, suy luận từng bước dựa trên tài liệu pháp luật được cung cấp, "
    "sau đó đưa ra câu trả lời chính xác có trích dẫn Điều, Khoản cụ thể. "
    "Nếu tài liệu không đủ thông tin, hãy nói rõ giới hạn của câu trả lời."
)

# Apply qwen3-thinking chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-thinking",
)

print(f"✅ Chat template applied: qwen3-thinking")

Model does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Chat template applied: qwen3-thinking


In [4]:
# ============================================================
# Load JSONL files
# ============================================================
def load_legal_jsonl(path: str) -> list[dict]:
    """Load legal JSONL file and return list of records."""
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                records.append(rec)
            except json.JSONDecodeError as e:
                print(f"⚠️ Skipping invalid JSON at line {line_num}: {e}")
    return records

train_records = load_legal_jsonl(TRAIN_PATH)
val_records   = load_legal_jsonl(VAL_PATH)

print(f"📊 Train records loaded: {len(train_records):,}")
print(f"📊 Val records loaded:   {len(val_records):,}")

# Quick stats
laws = set()
for r in train_records:
    meta = r.get("meta", {})
    if meta.get("law"):
        laws.add(meta["law"])
print(f"📋 Unique laws covered: {len(laws)}")
print(f"   Laws: {sorted(laws)}")

📊 Train records loaded: 4,800
📊 Val records loaded:   600
📋 Unique laws covered: 0
   Laws: []


In [5]:
# ============================================================
# Convert legal records → conversation format
# ============================================================
THINK_BLOCK_RE = re.compile(r"<think>.*?</think>", flags=re.DOTALL)

def normalize_assistant_output(text: str) -> str:
    """Ensure assistant output follows <think>...</think>\n... format."""
    text = text.strip()
    if not text:
        return "<think></think>\n"

    m = THINK_BLOCK_RE.search(text)
    if m:
        think_block = m.group(0).strip()
        rest = text[m.end():].lstrip()
        return f"{think_block}\n{rest}".rstrip() if rest else f"{think_block}\n"
    else:
        # No <think> tags found — wrap the whole thing
        return f"<think></think>\n{text}".rstrip()


def build_user_message(instruction: str, context: str) -> str:
    """Combine RAG context + question into user message."""
    if context and context.strip():
        return (
            f"Tài liệu tham khảo:\n"
            f"{context.strip()}\n\n"
            f"Câu hỏi: {instruction.strip()}"
        )
    else:
        return instruction.strip()


def convert_records_to_conversations(records: list[dict]) -> list[dict]:
    """Convert legal records into conversation format for SFT."""
    conversations_list = []
    skipped = 0

    for rec in records:
        instruction = rec.get("instruction", "").strip()
        context     = rec.get("input", "").strip()
        output      = rec.get("output", "").strip()

        # Skip invalid records
        if not instruction or not output:
            skipped += 1
            continue

        user_msg = build_user_message(instruction, context)
        assistant_msg = normalize_assistant_output(output)

        conversation = [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_msg},
            {"role": "assistant", "content": assistant_msg},
        ]
        conversations_list.append({"conversations": conversation})

    if skipped > 0:
        print(f"⚠️ Skipped {skipped} invalid records")

    return conversations_list


print("🔄 Converting train records to conversation format...")
train_convos = convert_records_to_conversations(train_records)
print(f"   ✅ Train conversations: {len(train_convos):,}")

print("🔄 Converting val records to conversation format...")
val_convos = convert_records_to_conversations(val_records)
print(f"   ✅ Val conversations: {len(val_convos):,}")

🔄 Converting train records to conversation format...
   ✅ Train conversations: 4,800
🔄 Converting val records to conversation format...
   ✅ Val conversations: 600


In [6]:
# ============================================================
# Preview a sample conversation
# ============================================================
sample = train_convos[0]["conversations"]
print("=" * 80)
print("📝 SAMPLE CONVERSATION")
print("=" * 80)
for msg in sample:
    role = msg["role"].upper()
    content = msg["content"]
    if len(content) > 500:
        content = content[:500] + "...[truncated]"
    print(f"\n--- {role} ---")
    print(content)
print("\n" + "=" * 80)

📝 SAMPLE CONVERSATION

--- SYSTEM ---
Bạn là một trợ lý pháp lý chuyên về luật Việt Nam. Hãy phân tích câu hỏi, suy luận từng bước dựa trên tài liệu pháp luật được cung cấp, sau đó đưa ra câu trả lời chính xác có trích dẫn Điều, Khoản cụ thể. Nếu tài liệu không đủ thông tin, hãy nói rõ giới hạn của câu trả lời.

--- USER ---
Tài liệu tham khảo:
TÌNH HUỐNG:
Công ty TNHH Thương mại Dịch vụ XYZ (gọi tắt là Công ty XYZ) ký hợp đồng lao động không xác định thời hạn với bà Nguyễn Thị Lan vào ngày 01/01/2018, với vị trí Trưởng phòng Kinh doanh. Mức lương thỏa thuận là 20.000.000 VNĐ/tháng. Đến ngày 15/03/2023, do tình hình kinh doanh khó khăn, Công ty XYZ quyết định tái cơ cấu bộ máy, trong đó có việc cắt giảm vị trí Trưởng phòng Kinh doanh. Công ty XYZ thông báo cho bà Lan về việc chấm dứt hợp đồng lao động với lý do cơ c...[truncated]

--- ASSISTANT ---
<think>
1. Vấn đề pháp lý mấu chốt cần phân tích là liệu việc Công ty XYZ chậm thanh toán lương tháng 02/2023 cho bà Lan có vi phạm quy địn

In [7]:
# ============================================================
# Create HuggingFace Datasets + Apply Chat Template
# ============================================================
train_dataset = Dataset.from_list(train_convos)
val_dataset   = Dataset.from_list(val_convos)

print(f"📊 Train dataset: {len(train_dataset):,} samples")
print(f"📊 Val dataset:   {len(val_dataset):,} samples")

# Apply chat template to generate the 'text' field
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        for convo in convos
    ]
    return {"text": texts}

print("\n🔄 Applying qwen3-thinking chat template...")
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset   = val_dataset.map(formatting_prompts_func, batched=True)
print("   ✅ Chat template applied")

📊 Train dataset: 4,800 samples
📊 Val dataset:   600 samples

🔄 Applying qwen3-thinking chat template...


Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

   ✅ Chat template applied


In [8]:
# ============================================================
# Filter by token length
# ============================================================
num_proc = mp.cpu_count()
print(f"🔧 Using {num_proc} CPU cores for filtering")
print(f"🔧 Max context window: {MAX_CONTEXT_WINDOW} tokens")

_text_tok = getattr(tokenizer, "tokenizer", tokenizer)

def filter_long_sequences(examples):
    texts = examples["text"]
    tokenized = _text_tok(
        texts,
        truncation=False,
        padding=False,
        add_special_tokens=False,
    )["input_ids"]
    return [len(toks) <= MAX_CONTEXT_WINDOW for toks in tokenized]

# Filter train
before = len(train_dataset)
train_dataset = train_dataset.filter(filter_long_sequences, batched=True, num_proc=num_proc)
after = len(train_dataset)
print(f"\n📊 Train: {before} → {after} (removed {before - after} samples > {MAX_CONTEXT_WINDOW} tokens)")

# Filter val
before = len(val_dataset)
val_dataset = val_dataset.filter(filter_long_sequences, batched=True, num_proc=num_proc)
after = len(val_dataset)
print(f"📊 Val:   {before} → {after} (removed {before - after} samples > {MAX_CONTEXT_WINDOW} tokens)")

🔧 Using 86 CPU cores for filtering
🔧 Max context window: 8192 tokens


Filter (num_proc=86):   0%|          | 0/4800 [00:00<?, ? examples/s]


📊 Train: 4800 → 4800 (removed 0 samples > 8192 tokens)


Filter (num_proc=86):   0%|          | 0/600 [00:00<?, ? examples/s]

📊 Val:   600 → 600 (removed 0 samples > 8192 tokens)


In [9]:
# ============================================================
# Validate assistant format: <think>...</think>\n...
# ============================================================
def check_assistant_format(examples):
    convos = examples["conversations"]
    ok = []
    for convo in convos:
        good = True
        for m in convo:
            if m["role"] == "assistant":
                c = m.get("content", "")
                if "<think>" not in c or "</think>" not in c:
                    good = False
                    break
                if not re.search(r"</think>\n", c):
                    good = False
                    break
        ok.append(good)
    return {"_ok": ok}

check = train_dataset.map(
    check_assistant_format,
    batched=True,
    remove_columns=train_dataset.column_names,
    num_proc=num_proc,
)
bad = len(check) - sum(check["_ok"])
print(f"\n🔍 Format verification: {bad} samples with invalid <think>...</think> format")

if bad > 0:
    print(f"   Filtering out {bad} malformed samples...")
    train_dataset = train_dataset.filter(lambda x: all(
        (m["role"] != "assistant") or (("<think>" in m["content"]) and ("</think>\n" in m["content"]))
        for m in x["conversations"]
    ))
    print(f"   ✅ Train dataset after format filter: {len(train_dataset):,}")
else:
    print("   ✅ All samples have valid format")

Map (num_proc=86):   0%|          | 0/4800 [00:00<?, ? examples/s]


🔍 Format verification: 0 samples with invalid <think>...</think> format
   ✅ All samples have valid format


In [10]:
# ============================================================
# Token length statistics
# ============================================================
def get_token_lengths(dataset, sample_size=500):
    """Get token length distribution for the dataset."""
    import random
    indices = random.sample(range(len(dataset)), min(sample_size, len(dataset)))
    lengths = []
    for i in indices:
        tokens = _text_tok(
            dataset[i]["text"],
            truncation=False,
            padding=False,
            add_special_tokens=False,
        )["input_ids"]
        lengths.append(len(tokens))
    return lengths

lengths = get_token_lengths(train_dataset)
print(f"\n📊 Token length statistics (sampled {len(lengths)} records):")
print(f"   Min:    {min(lengths):,}")
print(f"   Max:    {max(lengths):,}")
print(f"   Mean:   {sum(lengths)/len(lengths):,.0f}")
print(f"   Median: {sorted(lengths)[len(lengths)//2]:,}")
print(f"   P90:    {sorted(lengths)[int(len(lengths)*0.9)]:,}")
print(f"   P95:    {sorted(lengths)[int(len(lengths)*0.95)]:,}")


📊 Token length statistics (sampled 500 records):
   Min:    858
   Max:    5,565
   Mean:   3,248
   Median: 3,234
   P90:    4,394
   P95:    4,654


In [11]:
# ============================================================
# Final preview: see what the model will actually see
# ============================================================
print("\n" + "=" * 80)
print("📝 FINAL FORMATTED TEXT (first 1200 chars):")
print("=" * 80)
print(train_dataset[0]["text"][:1200])
print("...")
print("=" * 80)


📝 FINAL FORMATTED TEXT (first 1200 chars):
<|im_start|>system
Bạn là một trợ lý pháp lý chuyên về luật Việt Nam. Hãy phân tích câu hỏi, suy luận từng bước dựa trên tài liệu pháp luật được cung cấp, sau đó đưa ra câu trả lời chính xác có trích dẫn Điều, Khoản cụ thể. Nếu tài liệu không đủ thông tin, hãy nói rõ giới hạn của câu trả lời.<|im_end|>
<|im_start|>user
Tài liệu tham khảo:
TÌNH HUỐNG:
Công ty TNHH Thương mại Dịch vụ XYZ (gọi tắt là Công ty XYZ) ký hợp đồng lao động không xác định thời hạn với bà Nguyễn Thị Lan vào ngày 01/01/2018, với vị trí Trưởng phòng Kinh doanh. Mức lương thỏa thuận là 20.000.000 VNĐ/tháng. Đến ngày 15/03/2023, do tình hình kinh doanh khó khăn, Công ty XYZ quyết định tái cơ cấu bộ máy, trong đó có việc cắt giảm vị trí Trưởng phòng Kinh doanh. Công ty XYZ thông báo cho bà Lan về việc chấm dứt hợp đồng lao động với lý do cơ cấu lại sản xuất, kinh doanh theo Điều 42 Bộ luật Lao động 2019, và đề nghị bà Lan nhận trợ cấp mất việc làm theo quy định. Tuy nhiên, b

## 6. 🏋️ Training Configuration

### Hyperparameters được tối ưu cho legal domain:

| Param | Value | Lý do |
|---|---|---|
| `per_device_train_batch_size` | 2 | Cân bằng VRAM/speed trên A100 |
| `gradient_accumulation_steps` | 4 | Effective batch = 8 |
| `learning_rate` | 2e-4 | Standard cho QLoRA SFT |
| `num_train_epochs` | 3 | 3 epochs cho ~7.6k samples |
| `warmup_ratio` | 0.05 | 5% warmup steps |
| `lr_scheduler_type` | cosine | Smooth decay |
| `train_on_responses_only` | True | Chỉ tính loss trên assistant response |

In [16]:
from trl import SFTTrainer, SFTConfig

# Output directory for checkpoints
output_dir = "/mnt/v/legal_qwen35_output_advanced"
os.makedirs(output_dir, exist_ok=True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # Effective batch size = 8
        warmup_ratio = 0.05,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        save_steps = 200,
        save_total_limit = 3,              # Keep last 3 checkpoints
        save_strategy = "steps",
        eval_strategy = "steps",
        eval_steps = 200,                  # Evaluate every 200 steps
        load_best_model_at_end = True,     # Load best checkpoint at end
        metric_for_best_model = "eval_loss",
        greater_is_better = False,
        report_to = "none",                # Đổi thành "wandb" nếu muốn tracking
        output_dir = output_dir,                     # Dùng bfloat16 trên A100/H100
        max_seq_length = max_seq_length,
    ),
)

print(f"\n✅ Trainer initialized")
print(f"   Effective batch size: {2 * 4} = {2} × {4}")
total_steps = (len(train_dataset) // (2 * 4)) * 3
print(f"   Estimated total steps: ~{total_steps:,}")
print(f"   Estimated training time: ~{total_steps * 3 / 3600:.1f} hours (rough estimate)")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/4800 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/600 [00:00<?, ? examples/s]

[accelerate.utils.other|WARNING]Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.



✅ Trainer initialized
   Effective batch size: 8 = 2 × 4
   Estimated total steps: ~1,800
   Estimated training time: ~1.5 hours (rough estimate)


## 7. 🎭 Train on Responses Only

**Cực kỳ quan trọng**: Masking để model chỉ học phần assistant response,
không tính loss trên system prompt hay user message.

```
<|im_start|>system
Bạn là trợ lý pháp lý...           ← MASKED (label = -100)
<|im_end|>
<|im_start|>user
Tài liệu tham khảo: ...            ← MASKED (label = -100)
<|im_end|>
<|im_start|>assistant
<think>                             ← TRAINED ✅
1. Xác định vấn đề pháp lý...      ← TRAINED ✅
</think>                            ← TRAINED ✅
Câu trả lời chi tiết...            ← TRAINED ✅
<|im_end|>                          ← TRAINED ✅
```

In [17]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n<think>",
)

print("✅ train_on_responses_only applied")
print("   Only assistant responses (including <think> reasoning) will be trained")

Map (num_proc=64):   0%|          | 0/4800 [00:00<?, ? examples/s]

Filter (num_proc=64):   0%|          | 0/4800 [00:00<?, ? examples/s]

Map (num_proc=64):   0%|          | 0/600 [00:00<?, ? examples/s]

Filter (num_proc=64):   0%|          | 0/600 [00:00<?, ? examples/s]

✅ train_on_responses_only applied
   Only assistant responses (including <think> reasoning) will be trained


In [20]:
# ============================================================
# Verify masking is correct
# ============================================================
print("🔍 Verifying label masking on sample 0:")
print("")

sample_ids = trainer.train_dataset[0]["input_ids"]
sample_labels = trainer.train_dataset[0]["labels"]

# Show what is masked vs trained
decoded_input = tokenizer.decode(sample_ids)
decoded_labels = tokenizer.decode(
    [tokenizer.pad_token_id if x == -100 else x for x in sample_labels]
).replace(tokenizer.pad_token, "▪")

print("--- INPUT (first 500 chars) ---")
print(decoded_input[:500])
print("\n--- LABELS (▪ = masked, text = trained) first 500 chars ---")
print(decoded_labels[:500])

# Count masked vs trained tokens
masked = sum(1 for x in sample_labels if x == -100)
trained = sum(1 for x in sample_labels if x != -100)
print(f"\n📊 Token breakdown: {masked} masked + {trained} trained = {masked+trained} total")
print(f"   Training ratio: {100*trained/(masked+trained):.1f}% of tokens")

🔍 Verifying label masking on sample 0:

--- INPUT (first 500 chars) ---
<|im_start|>system
Bạn là một trợ lý pháp lý chuyên về luật Việt Nam. Hãy phân tích câu hỏi, suy luận từng bước dựa trên tài liệu pháp luật được cung cấp, sau đó đưa ra câu trả lời chính xác có trích dẫn Điều, Khoản cụ thể. Nếu tài liệu không đủ thông tin, hãy nói rõ giới hạn của câu trả lời.<|im_end|>
<|im_start|>user
Tài liệu tham khảo:
TÌNH HUỐNG:
Công ty TNHH Thương mại Dịch vụ XYZ (gọi tắt là Công ty XYZ) ký hợp đồng lao động không xác định thời hạn với bà Nguyễn Thị Lan vào ngày 01/01/2018

--- LABELS (▪ = masked, text = trained) first 500 chars ---
▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪

## 8. 🚂 Start Training

In [21]:
# GPU memory before training
gpu_stats = torch.cuda.get_device_properties(0)
allocated = torch.cuda.max_memory_allocated() / 1024**3
reserved  = torch.cuda.max_memory_reserved() / 1024**3
print(f"🖥️ GPU: {gpu_stats.name}")
print(f"   Total VRAM:     {gpu_stats.total_mem / 1024**3:.1f} GB")
print(f"   Max Allocated:  {allocated:.1f} GB")
print(f"   Max Reserved:   {reserved:.1f} GB")
print(f"\n🚀 Starting training...")
print(f"   Dataset: {len(train_dataset):,} train / {len(val_dataset):,} val")
print(f"   Epochs: 3")
print(f"   This will take approximately 2-5 hours on A100.")
print("")

🖥️ GPU: NVIDIA H200


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

In [ ]:
# Compilation can take 2-3 minutes, so please be patient
trainer_stats = trainer.train()

# If training is interrupted, resume with:
# trainer_stats = trainer.train(resume_from_checkpoint=True)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'bos_token_id': None, 'pad_token_id': 248077}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,800 | Num Epochs = 3 | Total steps = 1,800
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 2,859,008,000 of 37,966,189,936 (7.53% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
200,0.528824,0.514079
400,0.492438,0.488579
600,0.468549,0.474832
800,0.324273,0.483190
1000,0.328459,0.474524
1200,0.322363,0.466864


Unsloth: Restored added_tokens_decoder metadata in /mnt/v/legal_qwen35_output_advanced/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /mnt/v/legal_qwen35_output_advanced/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /mnt/v/legal_qwen35_output_advanced/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /mnt/v/legal_qwen35_output_advanced/checkpoint-800/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /mnt/v/legal_qwen35_output_advanced/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /mnt/v/legal_qwen35_output_advanced/checkpoint-1200/tokenizer_config.json.


In [18]:
# ============================================================
# Training summary
# ============================================================
print("\n" + "=" * 80)
print("📊 TRAINING COMPLETE")
print("=" * 80)
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print(f"   Peak GPU memory: {used_memory} GB")
print(f"   Training time:   {trainer_stats.metrics.get('train_runtime', 0)/60:.1f} minutes")
print(f"   Final loss:      {trainer_stats.metrics.get('train_loss', 'N/A')}")
print("=" * 80)


📊 TRAINING COMPLETE
   Peak GPU memory: 132.78 GB
   Training time:   192.5 minutes
   Final loss:      0.2508146463264346


## 9. 🧪 Test Inference

Thử model fine-tuned với một câu hỏi pháp lý thực tế.

In [19]:
# Switch to inference mode
FastLanguageModel.for_inference(model)

# Test with a legal question
test_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {
        "role": "user",
        "content": (
            "Tài liệu tham khảo:\n"
            "[#1 | Luật 91/2015/QH13 | Điều 1 | nguồn: vector]\n"
            "Điều 1. Phạm vi điều chỉnh\n"
            "Bộ luật này quy định địa vị pháp lý, chuẩn mực pháp lý về cách ứng xử "
            "của cá nhân, pháp nhân; quyền, nghĩa vụ về nhân thân và tài sản của cá nhân, "
            "pháp nhân trong các quan hệ được hình thành trên cơ sở bình đẳng, tự do ý chí, "
            "độc lập về tài sản và tự chịu trách nhiệm.\n\n"
            "Câu hỏi: Bộ luật Dân sự 2015 điều chỉnh những quan hệ nào?"
        ),
    },
]

input_ids = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

print("🤖 Generating response...\n")

output = model.generate(
    input_ids=input_ids,
    max_new_tokens=2048,
    temperature=0.6,
    top_p=0.95,
    do_sample=True,
)

response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)

print("=" * 80)
print("📝 MODEL RESPONSE:")
print("=" * 80)
print(response)
print("=" * 80)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


🤖 Generating response...

📝 MODEL RESPONSE:
1. Xác định vấn đề pháp lý: Câu hỏi yêu cầu xác định phạm vi các quan hệ mà Bộ luật Dân sự 2015 điều chỉnh.
2. Tìm điều khoản liên quan: Đối chiếu với tài liệu tham khảo, Điều 1 của Luật 91/2015/QH13 (Bộ luật Dân sự 2015) có tiêu đề "Phạm vi điều chỉnh" là điều khoản trực tiếp trả lời câu hỏi này.
3. Trích xuất nội dung: Điều 1 quy định Bộ luật này điều chỉnh "địa vị pháp lý, chuẩn mực pháp lý về cách ứng xử của cá nhân, pháp nhân; quyền, nghĩa vụ về nhân thân và tài sản của cá nhân, pháp nhân trong các quan hệ được hình thành trên cơ sở bình đẳng, tự do ý chí, độc lập về tài sản và tự chịu trách nhiệm."
4. Tổng hợp và kết luận: Dựa trên Điều 1, Bộ luật Dân sự 2015 điều chỉnh các quan hệ liên quan đến địa vị pháp lý, chuẩn mực ứng xử, quyền và nghĩa vụ về nhân thân, tài sản của cá nhân và pháp nhân, trong các quan hệ được hình thành trên cơ sở bình đẳng, tự do ý chí, độc lập về tài sản và tự chịu trách nhiệm.
</think>

Theo Điều 1 của Luật số

## 10. 💾 Save Model

### 10a. Save LoRA adapter (nhẹ, ~100MB)

In [20]:
# Save LoRA adapter locally
lora_dir = "./legal_qwen35_lora"
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)
print(f"✅ LoRA adapter saved to {lora_dir}")

# Optional: Push to HuggingFace Hub
# model.push_to_hub("your-username/legal-qwen35-lora", token="YOUR_HF_TOKEN")
# tokenizer.push_to_hub("your-username/legal-qwen35-lora", token="YOUR_HF_TOKEN")

Unsloth: Restored added_tokens_decoder metadata in ./legal_qwen35_lora/tokenizer_config.json.


✅ LoRA adapter saved to ./legal_qwen35_lora


### 10b. Save merged 16-bit model (cho deployment chất lượng cao)

In [21]:
# Optional: Save merged 16-bit model
# Uncomment if you have enough disk space (~40-70 GB)

# merged_dir = "./legal_qwen35_merged_16bit"
# model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
# print(f"✅ Merged 16-bit model saved to {merged_dir}")

# Or push to HuggingFace Hub directly:
# model.push_to_hub_merged(
#     "your-username/legal-qwen35-35b-a3b-sft",
#     tokenizer,
#     save_method="merged_16bit",
#     token="YOUR_HF_TOKEN",
# )

print("ℹ️ Uncomment the code above to save merged 16-bit model")

ℹ️ Uncomment the code above to save merged 16-bit model


### 10c. Export GGUF (cho inference với llama.cpp / Ollama)

In [22]:
# Export GGUF quantized model
# Choose quantization methods based on your deployment needs:
#   q4_k_m  → smallest, ~4-bit, good for limited hardware
#   q5_k_m  → balanced quality/size
#   q8_0    → highest quality quantization
#   f16     → full half-precision

# Uncomment to export:
# model.push_to_hub_gguf(
#     "your-username/legal-qwen35-35b-a3b-sft-GGUF",
#     tokenizer,
#     quantization_method=["q4_k_m", "q8_0"],
#     token="YOUR_HF_TOKEN",
# )

# Or save locally:
# model.save_pretrained_gguf(
#     "./legal_qwen35_gguf",
#     tokenizer,
#     quantization_method="q8_0",
# )

print("ℹ️ Uncomment the code above to export GGUF model")

ℹ️ Uncomment the code above to export GGUF model


## 11. 📋 Training Checklist & Notes

### Trước khi chạy:
- [ ] Upload `data_gen/train.jsonl` và `data_gen/val.jsonl` lên cloud environment
- [ ] Kiểm tra GPU: cần A100 40GB+ hoặc H100
- [ ] Cập nhật `TRAIN_PATH` và `VAL_PATH` nếu đường dẫn khác
- [ ] (Optional) Setup WandB: đổi `report_to = "wandb"` và login

### Sau khi train:
- [ ] Kiểm tra loss curve — nên giảm dần và ổn định
- [ ] Test inference với vài câu hỏi pháp lý
- [ ] So sánh output với Gemini teacher (ground truth)
- [ ] Export GGUF nếu muốn deploy local

### Tuning tips:
- Nếu **loss không giảm**: giảm `learning_rate` xuống `5e-5`
- Nếu **overfitting** (val_loss tăng): giảm `num_train_epochs` xuống 2, tăng `weight_decay`
- Nếu **OOM**: giảm `per_device_train_batch_size` xuống 1, tăng `gradient_accumulation_steps` lên 8
- Nếu muốn **nhanh hơn**: tăng `per_device_train_batch_size` lên 4 (nếu đủ VRAM)

### Tích hợp vào RAG pipeline:
Sau khi có model GGUF, chạy inference qua Ollama:
```bash
ollama create legal-assistant -f Modelfile
ollama run legal-assistant
```
Rồi cập nhật `rag_engine.py` để gọi model local thay vì Gemini API.

In [1]:
!apt update -y
!apt install -y git build-essential cmake
!apt install -y libcublas-dev-12-8

Get:1 http://deb.debian.org/debian bookworm InRelease [151 kB]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/debian12/x86_64  InRelease [1581 B]
Get:3 http://deb.debian.org/debian bookworm-updates InRelease [55.4 kB]
Get:4 http://deb.debian.org/debian-security bookworm-security InRelease [48.0 kB]
Get:5 http://deb.debian.org/debian bookworm/main amd64 Packages [8790 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/debian12/x86_64  Packages [1978 kB]
Get:7 http://deb.debian.org/debian-security bookworm-security/main amd64 Packages [306 kB]
Fetched 11.3 MB in 3s (3336 kB/s)



134 packages can be upgraded. Run 'apt list --upgradable' to see them.
N: Repository 'http://deb.debian.org/debian bookworm InRelease' changed its 'Version' value from '12.11' to '12.14'



build-essential is already the newest version (12.9).
cmake is already the newest version (3.25.1-1).
The following additional packages will be installed:
  git-man
Suggested packages:
  gettex

In [2]:
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp

!rm -rf build
!cmake -B build -DGGML_CUDA=ON
!cmake --build build -j64 --config Release 

Cloning into 'llama.cpp'...
remote: Enumerating objects: 98185, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 98185 (delta 0), reused 0 (delta 0), pack-reused 98180 (from 2)
Receiving objects: 100% (98185/98185), 395.79 MiB | 30.46 MiB/s, done.
Resolving deltas: 100% (70178/70178), done.
Updating files: 100% (2948/2948), done.
/root/llama.cpp
-- The C compiler identification is GNU 12.2.0
-- The CXX compiler identification is GNU 12.2.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.39.5") 
-- The 

In [5]:
%uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
%uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
%uv pip install --no-build-isolation flash-linear-attention causal_conv1d
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    %uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
%uv pip install --no-deps --upgrade "torchao"

Using Python 3.12.6 environment at: /usr/local
Resolved 4 packages in 72ms
Audited 4 packages in 0.45ms
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 9ms
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Audited 2 packages in 16ms
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Resolved 2 packages in 21ms
Audited 2 packages in 0.10ms
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Resolved 1 package in 31ms
Audited 1 package in 0.10ms
Note: you may need to restart the kernel to use updated packages.


In [6]:
import json, platform, sys, torch
from urllib.request import urlopen

py   = f"cp{sys.version_info.major}{sys.version_info.minor}"
tv   = ".".join(torch.__version__.split("+")[0].split(".")[:2])
assert (cu := (torch.version.cuda or "").split(".")[0]), "CUDA-enabled PyTorch required."
abi  = "TRUE" if torch.compiled_with_cxx11_abi() else "FALSE"
plat = "linux_x86_64" if platform.machine().lower() in ("x86_64", "amd64") else "linux_aarch64"
whl  = f"{py}-{py}-{plat}.whl"

def find(repo, tag, match):
    api = f"https://api.github.com/repos/{repo}/releases/tags/{tag}"
    return next((a["browser_download_url"] for a in json.load(urlopen(api))["assets"] if match(a["name"])), None)

cc1d = find("Dao-AILab/causal-conv1d", "v1.6.1.post4",
            lambda n: n.endswith(whl) and f"+cu{cu}torch{tv}" in n and f"cxx11abi{abi}" in n)
assert cc1d, f"No causal-conv1d wheel for torch {torch.__version__}/cu{cu}/{py}/abi{abi}"
fla = find("fla-org/flash-linear-attention", "v0.4.2", lambda n: n.endswith(whl)) \
      or "https://github.com/fla-org/flash-linear-attention/archive/refs/tags/v0.4.2.tar.gz"

%uv pip install --no-deps "$cc1d" "$fla"
%uv pip install datasets peft bitsandbytes
%uv pip uninstall sentence-transformers torchcodec  # torchcodec import broken on Colab
%uv pip install z3-solver

Using Python 3.12.6 environment at: /usr/local
Audited 2 packages in 18ms
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Audited 3 packages in 23ms
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Resolved 1 package in 63ms
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages...

In [7]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 8192  # Cấu hình giống lúc train
checkpoint_path = "/mnt/v/legal_qwen35_output/checkpoint-1800"  # Thay đổi thành đường dẫn checkpoint của bạn (ví dụ: checkpoint-500)
print(f"🔄 Đang load model gốc và lora adapter từ: {checkpoint_path}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = checkpoint_path,
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    fast_inference = False,
)

model.push_to_hub("claspi2509/legal-qwen35-lora", token="your_hugging_face_token_here")
tokenizer.push_to_hub("claspi2509/legal-qwen35-lora", token="your_hugging_face_token_here")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🔄 Đang load model gốc và lora adapter từ: /mnt/v/legal_qwen35_output/checkpoint-1800...
==((====))==  Unsloth 2026.6.1: Fast Qwen3_5_MoE patching. Transformers: 5.2.0.
   \\   /|    NVIDIA H200. Num GPUs = 1. Max memory: 139.801 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu129. CUDA: 9.0. CUDA Toolkit: 12.9. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json:   0%|          | 0.00/187k [00:00<?, ?B/s]

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1026 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/876 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

HfHubHTTPError: (Request ID: Root=1-6a239599-65251eb742b0070c26a5b0ca;bb4653f5-86d4-417c-9b71-bd3d537a3fb6)

403 Forbidden: You don't have the rights to create a model under the namespace "your-username".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.